# End-to-End UWB Indoor Localisation Pipeline

This notebook demonstrates the complete **3-stage inference pipeline** for UWB signal path analysis.

```
Raw CIR Features
       │
       ▼
┌──────────────────────────────┐
│  Stage 1 — Classification    │
│  Gradient Boosting Classifier│
│  Output: NLOS_pred (0 or 1)  │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│  Stage 2 — Path 2 Label      │
│  PATH2_NLOS = 1 (always)     │
│  (per spec: if path 1 is LOS │
│   path 2 is NLOS; if path 1  │
│   is NLOS, path 2 is NLOS)   │
└──────────────┬───────────────┘
               │
               ▼
┌──────────────────────────────┐
│  Stage 3 — Regression        │
│  HistGradientBoosting        │
│  Features + NLOS_pred        │
│  Output: [RANGE, RANGE2]     │
└──────────────────────────────┘
```

## Key Design Decisions

- **Stage 1** uses the **Gradient Boosting Classifier** trained in `classification.ipynb` — selected for best balance of accuracy, precision and F1-score (~89.4% accuracy on held-out test data).
- **Stage 2** is deterministic: per the dataset specification, *if the first path is LOS the second path is NLOS; if the first path is NLOS the second path is also NLOS* — so `PATH2_NLOS = 1` always.
- **Stage 3** uses the **HistGradientBoosting Regressor** trained in `regression2.ipynb` — best RMSE/R² across both paths. The NLOS prediction from Stage 1 replaces the ground-truth NLOS column that was used at training time.

## Setup — Load Models and Data

In [7]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    mean_squared_error, mean_absolute_error, r2_score
)

# ── Load trained models ──────────────────────────────────────────────────────
clf = joblib.load("gradient_boosting_model.pkl")   # Stage 1: classifier
reg = joblib.load("hgb_regressor.pkl")             # Stage 3: regressor

print("Classifier:", type(clf).__name__)
print("Regressor :", type(reg).__name__)

# ── Load held-out test data ───────────────────────────────────────────────────
# Classification features and true labels
clf_data = np.load("classification_data.npz")
X_test_clf   = clf_data["X_test"]    # (8400, 103) — preprocessed features
y_test_nlos  = clf_data["y_test"]    # (8400,)     — true NLOS labels

# Regression true targets
reg_data = np.load("multi_regression_data.npz")
y_test_reg = reg_data["y_test"]      # (8400, 2)   — true [RANGE, RANGE2]

print(f"\nTest samples : {len(X_test_clf):,}")
print(f"Feature dims : {X_test_clf.shape[1]}")
print(f"Regression targets shape: {y_test_reg.shape}")

Classifier: GradientBoostingClassifier
Regressor : MultiOutputRegressor

Test samples : 8,400
Feature dims : 103
Regression targets shape: (8400, 2)


/home/xenonic/Documents/Github/CSC3105/.venv/lib/python3.14/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DummyClassifier from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/xenonic/Documents/Github/CSC3105/.venv/lib/python3.14/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/xenonic/Documents/Github/CSC3105/.venv/lib/python3.14/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpic

## Stage 1 — LOS/NLOS Classification

Run the Gradient Boosting classifier on the held-out test features to produce predicted NLOS labels.

In [8]:
# Stage 1 — predict NLOS label for Path 1
nlos_pred  = clf.predict(X_test_clf)           # (8400,) — predicted labels
nlos_prob  = clf.predict_proba(X_test_clf)[:, 1]  # P(NLOS)

acc  = accuracy_score(y_test_nlos, nlos_pred)
f1   = f1_score(y_test_nlos, nlos_pred)
prec = precision_score(y_test_nlos, nlos_pred)
rec  = recall_score(y_test_nlos, nlos_pred)

print("Stage 1 — Classification Results")
print("=" * 40)
print(f"Accuracy : {acc:.4f}")
print(f"F1       : {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print()
print(f"Predicted LOS  (0): {(nlos_pred == 0).sum():,}")
print(f"Predicted NLOS (1): {(nlos_pred == 1).sum():,}")

Stage 1 — Classification Results
Accuracy : 0.8937
F1       : 0.8903
Precision: 0.9196
Recall   : 0.8629

Predicted LOS  (0): 4,459
Predicted NLOS (1): 3,941


## Stage 2 — Path 2 Label Derivation

Per the dataset specification, the second dominant path is **always NLOS** regardless of whether
Path 1 is LOS or NLOS:

- If Path 1 is **LOS** → Path 2 is the next shortest path, which is **NLOS**
- If Path 1 is **NLOS** → Path 2 is also **NLOS**

This is a deterministic rule, not a model prediction.

In [9]:
# Stage 2 — derive Path 2 NLOS label (always 1 per spec)
path2_nlos = np.ones(len(nlos_pred), dtype=int)

print("Stage 2 — Path 2 Label Derivation")
print("=" * 40)
print(f"PATH2_NLOS = 1 for all {len(path2_nlos):,} samples (deterministic rule)")

Stage 2 — Path 2 Label Derivation
PATH2_NLOS = 1 for all 8,400 samples (deterministic rule)


## Stage 3 — Range Estimation

Build the regression feature matrix by replacing the last column (which was the ground-truth NLOS
label during training) with the **predicted** NLOS label from Stage 1. Then run the HGB regressor
to estimate `[RANGE, RANGE2]` for both paths.

In [10]:
# Stage 3 — build pipeline feature matrix with predicted NLOS, then regress
# X_test_clf is (8400, 103) — the 103 preprocessed features (no NLOS column)
# Append predicted NLOS as the 104th feature (matching training-time structure)
X_test_pipeline = np.hstack([X_test_clf, nlos_pred.reshape(-1, 1)])  # (8400, 104)

range_pred = reg.predict(X_test_pipeline)   # (8400, 2) — [RANGE_pred, RANGE2_pred]

# Metrics — pipeline (using predicted NLOS)
rmse1 = np.sqrt(mean_squared_error(y_test_reg[:, 0], range_pred[:, 0]))
mae1  = mean_absolute_error(y_test_reg[:, 0], range_pred[:, 0])
r21   = r2_score(y_test_reg[:, 0], range_pred[:, 0])

rmse2 = np.sqrt(mean_squared_error(y_test_reg[:, 1], range_pred[:, 1]))
mae2  = mean_absolute_error(y_test_reg[:, 1], range_pred[:, 1])
r22   = r2_score(y_test_reg[:, 1], range_pred[:, 1])

print("Stage 3 — Regression Results (using predicted NLOS)")
print("=" * 55)
print(f"{'Metric':<10} {'Path 1':>12} {'Path 2':>12}")
print("-" * 36)
print(f"{'RMSE':<10} {rmse1:>12.4f} {rmse2:>12.4f}")
print(f"{'MAE':<10} {mae1:>12.4f} {mae2:>12.4f}")
print(f"{'R²':<10} {r21:>12.4f} {r22:>12.4f}")

Stage 3 — Regression Results (using predicted NLOS)
Metric           Path 1       Path 2
------------------------------------
RMSE             1.2248       1.4259
MAE              0.9233       1.0596
R²               0.7271       0.7891


## Baseline Comparison

Compare pipeline performance (predicted NLOS) vs the regression notebook baseline (true NLOS).
The gap quantifies the cost of using predicted rather than ground-truth channel labels.

In [11]:
# Baseline: regression2.ipynb used true NLOS labels at test time
baseline_rmse1, baseline_rmse2 = 1.1392, 1.3526
baseline_r21,   baseline_r22   = 0.7639, 0.8102

print("Baseline vs Pipeline Comparison")
print("=" * 62)
print(f"{'Metric':<14} {'Baseline P1':>12} {'Pipeline P1':>12} {'Delta P1':>10}")
print("-" * 50)
print(f"{'RMSE':<14} {baseline_rmse1:>12.4f} {rmse1:>12.4f} {rmse1 - baseline_rmse1:>+10.4f}")
print(f"{'R²':<14} {baseline_r21:>12.4f} {r21:>12.4f} {r21 - baseline_r21:>+10.4f}")
print()
print(f"{'Metric':<14} {'Baseline P2':>12} {'Pipeline P2':>12} {'Delta P2':>10}")
print("-" * 50)
print(f"{'RMSE':<14} {baseline_rmse2:>12.4f} {rmse2:>12.4f} {rmse2 - baseline_rmse2:>+10.4f}")
print(f"{'R²':<14} {baseline_r22:>12.4f} {r22:>12.4f} {r22 - baseline_r22:>+10.4f}")
print()
print("Note: A positive delta means the pipeline performs slightly worse,")
print("which is expected — classifier errors propagate into the regressor.")

Baseline vs Pipeline Comparison
Metric          Baseline P1  Pipeline P1   Delta P1
--------------------------------------------------
RMSE                 1.1392       1.2248    +0.0856
R²                   0.7639       0.7271    -0.0368

Metric          Baseline P2  Pipeline P2   Delta P2
--------------------------------------------------
RMSE                 1.3526       1.4259    +0.0733
R²                   0.8102       0.7891    -0.0211

Note: A positive delta means the pipeline performs slightly worse,
which is expected — classifier errors propagate into the regressor.


## Sample Prediction Table

Inspect the first 10 test samples end-to-end: true vs predicted labels and distances.

In [12]:
n = 10
label_map = {0: 'LOS', 1: 'NLOS'}

rows = []
for i in range(n):
    rows.append({
        "sample_id"   : i,
        "true_NLOS"   : label_map[int(y_test_nlos[i])],
        "pred_NLOS"   : label_map[int(nlos_pred[i])],
        "PATH2_NLOS"  : label_map[path2_nlos[i]],
        "true_RANGE"  : round(float(y_test_reg[i, 0]), 3),
        "pred_RANGE"  : round(float(range_pred[i, 0]), 3),
        "true_RANGE2" : round(float(y_test_reg[i, 1]), 3),
        "pred_RANGE2" : round(float(range_pred[i, 1]), 3),
    })

df_results = pd.DataFrame(rows)
df_results["clf_correct"] = df_results["true_NLOS"] == df_results["pred_NLOS"]
df_results["err_RANGE"]   = (df_results["pred_RANGE"]  - df_results["true_RANGE"]).round(3)
df_results["err_RANGE2"]  = (df_results["pred_RANGE2"] - df_results["true_RANGE2"]).round(3)

df_results

,sample_id,true_NLOS,pred_NLOS,PATH2_NLOS,true_RANGE,pred_RANGE,true_RANGE2,pred_RANGE2,clf_correct,err_RANGE,err_RANGE2
0,0,LOS,LOS,NLOS,3.04,1.816,4.839,4.224,True,-1.224,-0.615
1,1,LOS,NLOS,NLOS,4.17,4.039,6.568,6.319,False,-0.131,-0.249
2,2,LOS,LOS,NLOS,3.00,2.970,6.298,5.917,True,-0.030,-0.381
3,3,NLOS,NLOS,NLOS,6.21,5.052,11.306,10.108,True,-1.158,-1.198
4,4,NLOS,NLOS,NLOS,3.39,4.359,4.589,6.851,True,0.969,2.262
5,5,LOS,LOS,NLOS,2.49,1.177,5.188,4.033,True,-1.313,-1.155
6,6,NLOS,NLOS,NLOS,4.20,4.691,6.598,8.110,True,0.491,1.512
7,7,NLOS,NLOS,NLOS,2.84,3.426,7.037,8.912,True,0.586,1.875
8,8,NLOS,NLOS,NLOS,4.42,3.349,11.015,10.737,True,-1.071,-0.278
9,9,LOS,LOS,NLOS,3.54,3.540,5.639,6.890,True,0.000,1.251
